In [1]:
import numpy as np
import pandas as pd
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from collections import Counter

# ============ 配置 ============
DATA_DIR = "data"
CLEAN_TRAIN = f"{DATA_DIR}/clean_train_v2.csv"
CLEAN_TEST = f"{DATA_DIR}/clean_test_v2.csv"
SAMPLE_SUB = f"{DATA_DIR}/sample_submission.csv"

print("📁 讀取數據...")
clean_train = pd.read_csv(CLEAN_TRAIN, low_memory=False)
clean_test = pd.read_csv(CLEAN_TEST, low_memory=False)
sample_sub = pd.read_csv(SAMPLE_SUB, low_memory=False)

print(f"clean_train: {clean_train.shape}")
print(f"clean_test: {clean_test.shape}")
print(f"sample_sub: {sample_sub.shape}")

📁 讀取數據...
clean_train: (82572, 39)
clean_test: (3084, 35)
sample_sub: (1001, 4)


In [2]:
# ============ 特徵定義 ============
KEY = 'rally_uid'
ACTION_COL = 'next_actionId'
POINT_COL = 'next_pointId'

# 尋找 Task3 目標欄位
RALLY_TARGET_CANDIDATES = ['serverGetPoint', 'server_won_point']
RALLY_TARGET = next((c for c in RALLY_TARGET_CANDIDATES if c in clean_train.columns), None)

# 群組欄位
GROUP_COL = 'match' if 'match' in clean_train.columns else (
    'rally_id' if 'rally_id' in clean_train.columns else None
)

print(f"\n✅ 標籤: {ACTION_COL}, {POINT_COL}, {RALLY_TARGET}")
print(f"✅ 群組: {GROUP_COL}")



✅ 標籤: next_actionId, next_pointId, serverGetPoint
✅ 群組: rally_id


In [3]:
# ============ 特徵選擇 ============
exclude_cols = [
    'match', 'match_id', 'rally_id', 'stroke_number', 'strickNumber',
    'next_actionId', 'next_pointId', 'serverGetPoint', 'server_won_point',
    'rally_uid', 'actionId', 'pointId',
    # 🔴 移除洩漏特徵
    'rally_length', 'rally_progress', 
    'is_early_rally', 'is_mid_rally', 'is_late_rally',
    'score_rally_interact',  # 基於 rally_progress
    'action_point_combo'  # 可能包含當前拍信息
]

# Task1 & Task2
features_task12 = [c for c in clean_train.columns if c not in exclude_cols]

# ============ Task3 特徵（修正版）============

# Task3 只排除「未來信息」，保留「已發生的信息」
exclude_task3 = [
    # 排除的欄位（標籤、ID等）
    'next_actionId', 'next_pointId', 'serverGetPoint', 'server_won_point',
    'rally_uid', 'match', 'match_id', 'rally_id', 'strickNumber',
    
    # 🔴 只排除這些「未來信息」
    'rally_length',           # 總拍數（未來）
    'rally_progress',         # 基於 rally_length
    'is_early_rally',         # 基於 rally_progress
    'is_mid_rally',           # 基於 rally_progress
    'is_late_rally',          # 基於 rally_progress
    'score_rally_interact',   # 基於 rally_progress
]

# Task3 可以使用所有其他特徵
features_task3 = [c for c in clean_train.columns if c not in exclude_task3]


print(f"\n📊 特徵統計:")
print(f"Task1&2 特徵數: {len(features_task12)}")
print(f"Task3 特徵數: {len(features_task3)}")


📊 特徵統計:
Task1&2 特徵數: 28
Task3 特徵數: 31


In [4]:
# ============ 交叉驗證函數 ============
def cv_multiclass_f1(X, y, groups=None, n_splits=5, random_state=42, 
                      class_weight=None, params=None, verbose=True):
    """多分類 Macro F1 交叉驗證"""
    scores = []
    
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(X, y, groups))
    else:
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(skf.split(X, y))
    
    if params is None:
        params = {
            'n_estimators': 800,
            'learning_rate': 0.03,
            'num_leaves': 127,
            'max_depth': -1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_samples': 20,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': random_state,
            'n_jobs': -1,
            'verbose': -1
        }
    
    for fold, (train_idx, val_idx) in enumerate(splits, 1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        # ✅ 修正：為這個 fold 動態計算 class_weight
        if class_weight is not None:
            # 只使用訓練集中實際出現的類別
            fold_class_weight = {k: v for k, v in class_weight.items() if k in y_tr.values}
        else:
            fold_class_weight = None
        
        model = LGBMClassifier(**params, class_weight=fold_class_weight)
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_va)
        score = f1_score(y_va, y_pred, average='macro')
        scores.append(score)
        
        if verbose:
            print(f"  Fold {fold}: {score:.4f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    if verbose:
        print(f"  Mean: {mean_score:.4f} ± {std_score:.4f}")
    
    return mean_score

def cv_binary_auc(X, y, groups=None, n_splits=5, random_state=42, 
                  params=None, verbose=True):
    """二分類 AUC 交叉驗證"""
    scores = []
    
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        splits = list(gkf.split(X, y, groups))
    else:
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        splits = list(skf.split(X, y))
    
    if params is None:
        params = {
            'n_estimators': 600,
            'learning_rate': 0.05,
            'num_leaves': 63,
            'max_depth': -1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state,
            'n_jobs': -1,
            'verbose': -1
        }
    
    for fold, (train_idx, val_idx) in enumerate(splits, 1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        y_prob = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, y_prob)
        scores.append(score)
        
        if verbose:
            print(f"  Fold {fold}: {score:.4f}")
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    if verbose:
        print(f"  Mean: {mean_score:.4f} ± {std_score:.4f}")
    
    return mean_score


In [5]:
# ============ Task 1: ActionId（優化版）============
print("\n" + "="*60)
print("🎯 Task 1: ActionId Prediction (Macro F1)")
print("="*60)

X_action = clean_train[features_task12].copy()
y_action = clean_train[ACTION_COL].astype(int)
groups = clean_train[GROUP_COL] if GROUP_COL else None

# 類別權重（平方根逆向權重，平衡效果較好）
cnt_action = Counter(y_action)
weight_action = {cls: 1.0 / np.sqrt(count) for cls, count in cnt_action.items()}

# ✅ 優化的 stroke_number 樣本權重
stroke_weights = np.ones(len(clean_train))
stroke_numbers = clean_train['stroke_number'].values

# 針對測試集分佈（平均 3.07）的權重策略
for i, s in enumerate(stroke_numbers):
    if s == 1:
        stroke_weights[i] = 1.8
    elif s == 2:
        stroke_weights[i] = 2.5  # 最接近測試集平均
    elif s == 3:
        stroke_weights[i] = 3.0  # 測試集核心區域
    elif s == 4:
        stroke_weights[i] = 2.0
    elif s == 5:
        stroke_weights[i] = 1.2
    elif s == 6:
        stroke_weights[i] = 0.9
    else:
        stroke_weights[i] = 0.7

# 如果有壓力特徵，進一步調整權重
if 'score_pressure' in clean_train.columns:
    pressure = clean_train['score_pressure'].values
    for i, p in enumerate(pressure):
        if abs(p) >= 2:  # 壓力大的時刻
            stroke_weights[i] *= 1.15

print(f"\n📊 樣本權重統計:")
for s in range(1, 8):
    if s < 7:
        mask = (stroke_numbers == s)
        if mask.sum() > 0:
            w = stroke_weights[mask].mean()
            print(f"  Stroke {s}: {mask.sum():6d} 樣本，平均權重 {w:.2f}x")
    else:
        mask = (stroke_numbers >= 7)
        if mask.sum() > 0:
            w = stroke_weights[mask].mean()
            print(f"  Stroke 7+: {mask.sum():6d} 樣本，平均權重 {w:.2f}x")

weighted_avg_stroke = (stroke_numbers * stroke_weights).sum() / stroke_weights.sum()
print(f"  加權後平均 stroke: {weighted_avg_stroke:.2f} (目標: 3.07)")

# 參數（略微調整）
params_action = {
    'n_estimators': 900,        # 稍減: 1000 → 900
    'learning_rate': 0.025,     # 稍增: 0.02 → 0.025
    'num_leaves': 110,          # 稍減: 127 → 110
    'max_depth': 9,             # 限制: -1 → 9
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 25,    # 增加: 20 → 25
    'reg_alpha': 0.12,          # 稍增: 0.1 → 0.12
    'reg_lambda': 0.12,         # 稍增: 0.1 → 0.12
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("\n開始交叉驗證...")
f1_action_cv = cv_multiclass_f1(
    X_action, y_action, groups, 
    n_splits=5, 
    class_weight=weight_action,
    params=params_action,
    verbose=True
)

print(f"\n📈 Task1 CV Score: {f1_action_cv:.4f}")

# 🆕 Ensemble：訓練 3 個不同種子的模型
print("\n🎯 訓練 Ensemble 模型（3個種子）...")
models_action = []

for seed in [42, 123, 456]:
    print(f"  訓練模型 seed={seed}...")
    params_seed = params_action.copy()
    params_seed['random_state'] = seed
    
    model = LGBMClassifier(**params_seed, class_weight=weight_action)
    model.fit(X_action, y_action, sample_weight=stroke_weights)
    models_action.append(model)
    print(f"    ✓ 完成")

print("✅ Task1 Ensemble 訓練完成")

# 保留第一個模型用於特徵重要性
model_action = models_action[0]

# 特徵重要性
importances = pd.Series(
    model_action.feature_importances_, 
    index=X_action.columns
).sort_values(ascending=False)
print("\n🔝 Top 20 重要特徵:")
print(importances.head(20))


🎯 Task 1: ActionId Prediction (Macro F1)

📊 樣本權重統計:
  Stroke 1:  15527 樣本，平均權重 1.80x
  Stroke 2:  14556 樣本，平均權重 2.50x
  Stroke 3:  12815 樣本，平均權重 3.00x
  Stroke 4:  10268 樣本，平均權重 2.00x
  Stroke 5:   7380 樣本，平均權重 1.20x
  Stroke 6:   5397 樣本，平均權重 0.90x
  Stroke 7+:  16629 樣本，平均權重 0.70x
  加權後平均 stroke: 3.38 (目標: 3.07)

開始交叉驗證...
  Fold 1: 0.3157
  Fold 2: 0.3187
  Fold 3: 0.3254
  Fold 4: 0.3091
  Fold 5: 0.3042
  Mean: 0.3146 ± 0.0074

📈 Task1 CV Score: 0.3146

🎯 訓練 Ensemble 模型（3個種子）...
  訓練模型 seed=42...
    ✓ 完成
  訓練模型 seed=123...
    ✓ 完成
  訓練模型 seed=456...
    ✓ 完成
✅ Task1 Ensemble 訓練完成

🔝 Top 20 重要特徵:
action_seq_3             154265
score_phase              147199
action_point_encoded     107139
point_transition         105470
score_diff                90681
score_sum                 82567
curr_action_point         78678
prev1_action_point        72042
hand_point                68939
prev2_pointId             62027
scoreOther                60915
action_transition         57225
prev2

In [6]:
# ============ Task 2: PointId（優化版）============
print("\n" + "="*60)
print("🎯 Task 2: PointId Prediction (Macro F1)")
print("="*60)

X_point = clean_train[features_task12].copy()
y_point = clean_train[POINT_COL].astype(int)

# 類別權重
cnt_point = Counter(y_point)
weight_point = {cls: 1.0 / np.sqrt(count) for cls, count in cnt_point.items()}

# ✅ stroke_number 樣本權重（與 Task1 相同）
stroke_weights = np.ones(len(clean_train))
stroke_numbers = clean_train['stroke_number'].values

for i, s in enumerate(stroke_numbers):
    if s == 1:
        stroke_weights[i] = 1.8
    elif s == 2:
        stroke_weights[i] = 2.5
    elif s == 3:
        stroke_weights[i] = 3.0
    elif s == 4:
        stroke_weights[i] = 2.0
    elif s == 5:
        stroke_weights[i] = 1.2
    elif s == 6:
        stroke_weights[i] = 0.9
    else:
        stroke_weights[i] = 0.7

if 'score_pressure' in clean_train.columns:
    pressure = clean_train['score_pressure'].values
    for i, p in enumerate(pressure):
        if abs(p) >= 2:
            stroke_weights[i] *= 1.15

print(f"✅ 使用與 Task1 相同的樣本權重策略")

# 參數（Task2 用稍深的樹，因為落點預測更複雜）
params_point = {
    'n_estimators': 800,
    'learning_rate': 0.03,
    'num_leaves': 110,          # 降低: 127 → 110
    'max_depth': 10,            # 限制: -1 → 10 (Task2 用更深)
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,    # Task2 保持較小，允許學習稀有落點
    'reg_alpha': 0.12,
    'reg_lambda': 0.12,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("\n開始交叉驗證...")
f1_point_cv = cv_multiclass_f1(
    X_point, y_point, groups,
    n_splits=5,
    class_weight=weight_point,
    params=params_point,
    verbose=True
)

print(f"\n📈 Task2 CV Score: {f1_point_cv:.4f}")

# 🆕 Ensemble：訓練 3 個模型
print("\n🎯 訓練 Ensemble 模型（3個種子）...")
models_point = []

for seed in [42, 123, 456]:
    print(f"  訓練模型 seed={seed}...")
    params_seed = params_point.copy()
    params_seed['random_state'] = seed
    
    model = LGBMClassifier(**params_seed, class_weight=weight_point)
    model.fit(X_point, y_point, sample_weight=stroke_weights)
    models_point.append(model)
    print(f"    ✓ 完成")

print("✅ Task2 Ensemble 訓練完成")
model_point = models_point[0]


🎯 Task 2: PointId Prediction (Macro F1)
✅ 使用與 Task1 相同的樣本權重策略

開始交叉驗證...
  Fold 1: 0.2549
  Fold 2: 0.2440
  Fold 3: 0.2609
  Fold 4: 0.2498
  Fold 5: 0.2409
  Mean: 0.2501 ± 0.0072

📈 Task2 CV Score: 0.2501

🎯 訓練 Ensemble 模型（3個種子）...
  訓練模型 seed=42...
    ✓ 完成
  訓練模型 seed=123...
    ✓ 完成
  訓練模型 seed=456...
    ✓ 完成
✅ Task2 Ensemble 訓練完成


In [7]:
# ============ Task 3: ServerGetPoint（可選加權）============
print("\n" + "="*60)
print("🎯 Task 3: ServerGetPoint Prediction")
print("="*60)

if RALLY_TARGET is not None:
    X_rally = clean_train[features_task3].copy().fillna(0)
    y_rally = clean_train[RALLY_TARGET].astype(int)
    
    # ✅ 可選：也可以加 stroke_number 權重
    stroke_weights = np.ones(len(clean_train))
    stroke_numbers = clean_train['stroke_number'].values
    
    for i, s in enumerate(stroke_numbers):
        if s <= 3:
            stroke_weights[i] = 2.0
        elif s == 4:
            stroke_weights[i] = 1.3
        else:
            stroke_weights[i] = 0.9
    
    # 參數
    params_rally = {
        'n_estimators': 800,
        'learning_rate': 0.03,
        'num_leaves': 63,
        'max_depth': -1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    print("開始交叉驗證...")
    auc_rally_cv = cv_binary_auc(
        X_rally, y_rally, groups,
        params=params_rally,
        verbose=True
    )
    
    print(f"\n📈 Task3 CV Score: {auc_rally_cv:.4f}")
    
    # 訓練最終模型（使用樣本權重）
    print("訓練最終模型（加入樣本權重）...")
    model_rally = LGBMClassifier(**params_rally)
    model_rally.fit(X_rally, y_rally, sample_weight=stroke_weights)
    print("✅ Task3 模型已使用樣本權重訓練")
else:
    auc_rally_cv = 0.5
    model_rally = None
    print("⚠️ 找不到 Task3 目標欄位")


🎯 Task 3: ServerGetPoint Prediction
開始交叉驗證...
  Fold 1: 0.6252
  Fold 2: 0.6436
  Fold 3: 0.6348
  Fold 4: 0.6333
  Fold 5: 0.6235
  Mean: 0.6321 ± 0.0073

📈 Task3 CV Score: 0.6321
訓練最終模型（加入樣本權重）...
✅ Task3 模型已使用樣本權重訓練


In [8]:
# ============ 整體評分 ============
print("\n" + "="*60)
print("📊 整體 CV 評分")
print("="*60)
print(f"Task1 (ActionId)  : {f1_action_cv:.4f}")
print(f"Task2 (PointId)   : {f1_point_cv:.4f}")
print(f"Task3 (ServerWin) : {auc_rally_cv:.4f}")

blended_score = 0.4 * f1_action_cv + 0.4 * f1_point_cv + 0.2 * auc_rally_cv
print(f"\n🎯 Blended Score: {blended_score:.4f}")


📊 整體 CV 評分
Task1 (ActionId)  : 0.3146
Task2 (PointId)   : 0.2501
Task3 (ServerWin) : 0.6321

🎯 Blended Score: 0.3523


In [9]:
# ============ 測試集處理與預測 ============
print("\n" + "="*60)
print("🔮 測試集預測")
print("="*60)

# 對齊測試集
test_df = clean_test.copy()
test_df[KEY] = test_df[KEY].astype(str)
sample_sub[KEY] = sample_sub[KEY].astype(str)

# 找出每個 rally_uid 的最後一擊
if 'stroke_number' in test_df.columns:
    dedup = (test_df
             .sort_values([KEY, 'stroke_number'])
             .groupby(KEY, as_index=False)
             .tail(1))
else:
    dedup = test_df.drop_duplicates(subset=[KEY], keep='last')

# 按 sample_sub 順序重排
test_aligned = (dedup.set_index(KEY)
                .reindex(sample_sub[KEY])
                .reset_index())

print(f"對齊後測試集: {len(test_aligned)} 行")

# 特徵準備
features_in_test = [f for f in features_task12 if f in test_aligned.columns]
X_test_action = test_aligned[features_in_test].fillna(0)
X_test_point = X_test_action.copy()

features_task3_in_test = [f for f in features_task3 if f in test_aligned.columns]
X_test_rally = test_aligned[features_task3_in_test].fillna(0)

# ============================================================
# 🎯 測試集預測（使用 Ensemble）
# ============================================================
print("\n" + "="*60)
print("🎯 測試集預測")
print("="*60)

# 準備測試集特徵
X_test_action = test_aligned[features_task12].copy()
X_test_point = test_aligned[features_task12].copy()
X_test_rally = test_aligned[features_task3].copy() if RALLY_TARGET else None

# Task1: ActionId（Ensemble 預測）
print("\n🔮 Task1 預測（Ensemble）...")
pred_action_probs_list = []
for i, model in enumerate(models_action, 1):
    probs = model.predict_proba(X_test_action)
    pred_action_probs_list.append(probs)
    print(f"  模型 {i}/3 完成")

# 平均概率
pred_action_probs_avg = np.mean(pred_action_probs_list, axis=0)
pred_action_indices = pred_action_probs_avg.argmax(axis=1)
pred_action = models_action[0].classes_[pred_action_indices]
print(f"✅ Task1 Ensemble 完成，預測 {len(pred_action)} 個樣本")

# Task2: PointId（Ensemble 預測）
print("\n🔮 Task2 預測（Ensemble）...")
pred_point_probs_list = []
for i, model in enumerate(models_point, 1):
    probs = model.predict_proba(X_test_point)
    pred_point_probs_list.append(probs)
    print(f"  模型 {i}/3 完成")

pred_point_probs_avg = np.mean(pred_point_probs_list, axis=0)
pred_point_indices = pred_point_probs_avg.argmax(axis=1)
pred_point = models_point[0].classes_[pred_point_indices]
print(f"✅ Task2 Ensemble 完成，預測 {len(pred_point)} 個樣本")

# Task3: ServerGetPoint（單模型）
if model_rally is not None:
    print("\n🔮 Task3 預測...")
    pred_rally_prob = model_rally.predict_proba(X_test_rally)[:, 1]
    print(f"✅ Task3 完成")
else:
    pred_rally_prob = np.full(len(test_aligned), 0.5)
    print("⚠️ Task3 跳過")

# 組合預測
print("\n📊 組合預測結果...")
test_aligned['pred_actionId'] = pred_action
test_aligned['pred_pointId'] = pred_point
test_aligned['pred_serverWin'] = pred_rally_prob

# 生成 submission
submission = test_aligned[['rally_uid']].copy()
submission['actionId'] = test_aligned['pred_actionId']
submission['pointId'] = test_aligned['pred_pointId']
submission['serverGetPoint'] = test_aligned['pred_serverWin']

# 檢查
print(f"\nSubmission 大小: {submission.shape}")
print(f"Rally 數量: {submission['rally_uid'].nunique()}")
print("\n預測分佈:")
print(f"  actionId: {submission['actionId'].value_counts().head()}")
print(f"  pointId: {submission['pointId'].value_counts().head()}")
print(f"  serverGetPoint: mean={submission['serverGetPoint'].mean():.3f}")

# 保存
# 生成帶時間戳的檔名
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"result/submission_improved_{ts}.csv"

# 保存 submission
submission.to_csv(out_path, index=False, float_format="%.10f")
print("✅ 已輸出：", out_path)


🔮 測試集預測
對齊後測試集: 1001 行

🎯 測試集預測

🔮 Task1 預測（Ensemble）...


  模型 1/3 完成
  模型 2/3 完成
  模型 3/3 完成
✅ Task1 Ensemble 完成，預測 1001 個樣本

🔮 Task2 預測（Ensemble）...
  模型 1/3 完成
  模型 2/3 完成
  模型 3/3 完成
✅ Task2 Ensemble 完成，預測 1001 個樣本

🔮 Task3 預測...
✅ Task3 完成

📊 組合預測結果...

Submission 大小: (1001, 4)
Rally 數量: 1001

預測分佈:
  actionId: actionId
 1     259
-1     186
 10    118
 13    111
 6      71
Name: count, dtype: int64
  pointId: pointId
 9    330
-1    226
 8    128
 7    101
 0     88
Name: count, dtype: int64
  serverGetPoint: mean=0.519
✅ 已輸出： result/submission_improved_20251116_205327.csv


In [10]:
submission = pd.read_csv('result/submission_improved_20251102_174052.csv')
print(f"Submission 大小: {submission.shape}")
print(f"唯一 rally: {submission['rally_uid'].nunique()}")
print(f"重複: {submission['rally_uid'].duplicated().sum()}")

Submission 大小: (1001, 4)
唯一 rally: 1001
重複: 0


In [11]:
import pandas as pd

# 讀取兩個版本
quick = pd.read_csv('submission_quick_test.csv')
full = pd.read_csv('result/submission_improved_20251102_175120.csv')

print("📊 預測對比:")
print(f"\nTask1 (actionId) -1 比例:")
print(f"  快速版: {(quick['actionId'] == -1).sum() / len(quick):.2%}")
print(f"  完整版: {(full['actionId'] == -1).sum() / len(full):.2%}")

print(f"\nTask2 (pointId) -1 比例:")
print(f"  快速版: {(quick['pointId'] == -1).sum() / len(quick):.2%}")
print(f"  完整版: {(full['pointId'] == -1).sum() / len(full):.2%}")

print(f"\nTask3 (serverGetPoint):")
print(f"  快速版: mean={quick['serverGetPoint'].mean():.3f}")
print(f"  完整版: mean={full['serverGetPoint'].mean():.3f}")

# 檢查一致性
merged = quick.merge(full, on='rally_uid', suffixes=('_q', '_f'))
action_match = (merged['actionId_q'] == merged['actionId_f']).sum() / len(merged)
point_match = (merged['pointId_q'] == merged['pointId_f']).sum() / len(merged)

print(f"\n🔍 預測一致性:")
print(f"  Task1: {action_match:.1%}")
print(f"  Task2: {point_match:.1%}")

# 檢查特徵
clean_train = pd.read_csv('data/clean_train_v2.csv')
print(f"\n📊 完整版特徵數: {len(clean_train.columns)}")

suspicious = ['action_diversity', 'point_diversity', 'avg_strength_so_far']
found = [f for f in suspicious if f in clean_train.columns]
if found:
    print(f"❌ 發現洩漏特徵: {found}")
else:
    print(f"✅ 沒有明顯洩漏特徵")

📊 預測對比:

Task1 (actionId) -1 比例:
  快速版: 18.78%
  完整版: 19.48%

Task2 (pointId) -1 比例:
  快速版: 23.38%
  完整版: 22.78%

Task3 (serverGetPoint):
  快速版: mean=0.517
  完整版: mean=0.525

🔍 預測一致性:
  Task1: 55.5%
  Task2: 48.8%

📊 完整版特徵數: 39
✅ 沒有明顯洩漏特徵


In [12]:
import pandas as pd

clean_train = pd.read_csv('data/clean_train_v2.csv')

print("="*60)
print("🔍 檢查訓練集標籤")
print("="*60)

print(f"\nnext_actionId 的值:")
print(clean_train['next_actionId'].value_counts().sort_index())

neg1_count = (clean_train['next_actionId'] == -1).sum()
print(f"\n-1 的數量: {neg1_count}")
print(f"-1 的比例: {neg1_count / len(clean_train):.2%}")

if neg1_count == 0:
    print("\n❌ 問題確認：訓練集沒有 -1 標籤！")
    print("   這就是為什麼模型不會預測結束拍")
else:
    print("\n✅ 訓練集有 -1 標籤")
    print("   問題可能在模型訓練階段")

🔍 檢查訓練集標籤

next_actionId 的值:
next_actionId
-1     15420
 0      1981
 1     14991
 2      6544
 3      1709
 4      2878
 5      4107
 6      6582
 7      1475
 8       200
 9       425
 10     9923
 11     4116
 12     3752
 13     7810
 14      569
 15       73
 16       13
 17        3
 18        1
Name: count, dtype: int64

-1 的數量: 15420
-1 的比例: 18.67%

✅ 訓練集有 -1 標籤
   問題可能在模型訓練階段
